# Raw

In [1]:
import numpy as np
import os

folder = "Raw"

sample_file = os.path.join(folder, os.listdir(folder)[0])

data = np.load(sample_file)

print("Sample shape:", data.shape)

Sample shape: (2560, 27)


In [4]:
import os
import pandas as pd

data_folder = "Raw"

files = []
labels = []
subjects = []

for fname in sorted(os.listdir(data_folder)):

    if not fname.endswith(".npy"):
        continue

    path = os.path.join(data_folder, fname)

    parts = fname.split("_")
    subject = parts[2]

    if "Left" in fname:
        label = 0
    else:
        label = 1

    files.append(path)
    labels.append(label)
    subjects.append(subject)

df = pd.DataFrame({
    "file": files,
    "label": labels,
    "subject": subjects
})

print("Total samples:", len(df))
print("Total subjects:", df.subject.nunique())

df.to_csv("dataset_metadata_raw.csv", index=False)

Total samples: 20792
Total subjects: 87


In [5]:
from sklearn.model_selection import GroupKFold
import pandas as pd

df = pd.read_csv("dataset_metadata_raw.csv")

X = df["file"].values
y = df["label"].values
groups = df["subject"].values

gkf = GroupKFold(n_splits=5)

for fold,(train_idx,val_idx) in enumerate(gkf.split(X,y,groups)):

    train_subjects = set(groups[train_idx])
    val_subjects = set(groups[val_idx])

    overlap = train_subjects.intersection(val_subjects)

    print("Fold",fold)
    print("Train subjects:",len(train_subjects))
    print("Val subjects:",len(val_subjects))
    print("Overlap:",len(overlap))

Fold 0
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 1
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 2
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 3
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 4
Train subjects: 70
Val subjects: 17
Overlap: 0


In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, BatchNormalization,
    AveragePooling2D, Dropout, SeparableConv2D, Flatten, Dense
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Model

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5

CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Raw"

best_params = {
    'f1_filters': 32,
    'depth_multiplier': 4,
    'dropout_rate': 0.341,
    'learning_rate': 0.001175,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================
df = pd.read_csv("dataset_metadata_raw.csv")
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================
class EEGGenerator(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # fix shape (2560,27) -> (27,2560)
            data = data.T
            data = data[:, :, np.newaxis]
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# EEGNet Model
# =========================
def build_eegnet(f1_filters, depth_multiplier, dropout_rate, learning_rate):
    input_layer = Input(shape=(CHANNELS, TIMEPOINTS, 1))

    # Block 1
    block1 = Conv2D(f1_filters, (1, 64), padding="same", use_bias=False)(input_layer)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D(
        (CHANNELS, 1),
        depth_multiplier=depth_multiplier,
        use_bias=False,
        depthwise_constraint=max_norm(1.)
    )(block1)
    block1 = BatchNormalization()(block1)
    block1 = tf.keras.layers.Activation("elu")(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropout_rate)(block1)

    # Block 2
    f2_filters = f1_filters * depth_multiplier
    block2 = SeparableConv2D(f2_filters, (1, 16), use_bias=False, padding="same")(block1)
    block2 = BatchNormalization()(block2)
    block2 = tf.keras.layers.Activation("elu")(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropout_rate)(block2)

    # Classification layer
    flatten = Flatten()(block2)
    dense = Dense(N_CLASSES, activation="softmax", kernel_constraint=max_norm(0.25))(flatten)

    model = Model(inputs=input_layer, outputs=dense)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator(val_df, best_params['batch_size'], shuffle=False)

    model = build_eegnet(
        best_params['f1_filters'],
        best_params['depth_multiplier'],
        best_params['dropout_rate'],
        best_params['learning_rate']
    )

    # filenames include dataset name
    checkpoint_path = f"models/eegnet_{DATASET_NAME}_fold{fold + 1}.h5"
    
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/eegnet_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
515/515 [==============================] - 183s 348ms/step - loss: 0.5372 - accuracy: 0.7376 - val_loss: 0.5012 - val_accuracy: 0.8015 - lr: 0.0012
Epoch 2/50
515/515 [==============================] - 30s 58ms/step - loss: 0.4675 - accuracy: 0.7897 - val_loss: 0.4918 - val_accuracy: 0.7980 - lr: 0.0012
Epoch 3/50
515/515 [==============================] - 31s 59ms/step - loss: 0.4531 - accuracy: 0.7959 - val_loss: 0.5225 - val_accuracy: 0.8029 - lr: 0.0012
Epoch 4/50
515/515 [==============================] - 31s 59ms/step - loss: 0.4443 - accuracy: 0.7982 - val_loss: 0.4789 - val_accuracy: 0.8177 - lr: 0.0012
Epoch 5/50
515/515 [==============================] - 28s 55ms/step - loss: 0.4309 - accuracy: 0.8074 - val_loss: 0.5169 - val_accuracy: 0.7822 - lr: 0.0012
Epoch 6/50
515/515 [==============================] - 31s 60ms/step - loss: 0.4329 - accuracy: 0.8064 - val_loss: 0.4693 -

# Filtered 8-30

In [2]:
import numpy as np
import os

folder = "Filtered_8_30Hz"

sample_file = os.path.join(folder, os.listdir(folder)[0])

data = np.load(sample_file)

print("Sample shape:", data.shape)

Sample shape: (2560, 27)


In [1]:
import os
import pandas as pd

data_folder = "Filtered_8_30Hz"

files = []
labels = []
subjects = []

for fname in os.listdir(data_folder):

    if not fname.endswith(".npy"):
        continue

    path = os.path.join(data_folder, fname)

    parts = fname.split("_")
    subject = parts[2]

    if "Left" in fname:
        label = 0
    else:
        label = 1

    files.append(path)
    labels.append(label)
    subjects.append(subject)

df = pd.DataFrame({
    "file": files,
    "label": labels,
    "subject": subjects
})

print("Total samples:", len(df))
print("Total subjects:", df.subject.nunique())

df.to_csv("dataset_metadata.csv", index=False)


Total samples: 20792
Total subjects: 87


In [2]:
from sklearn.model_selection import GroupKFold
import pandas as pd

df = pd.read_csv("dataset_metadata.csv")

X = df["file"].values
y = df["label"].values
groups = df["subject"].values

gkf = GroupKFold(n_splits=5)

for fold,(train_idx,val_idx) in enumerate(gkf.split(X,y,groups)):

    train_subjects = set(groups[train_idx])
    val_subjects = set(groups[val_idx])

    overlap = train_subjects.intersection(val_subjects)

    print("Fold",fold)
    print("Train subjects:",len(train_subjects))
    print("Val subjects:",len(val_subjects))
    print("Overlap:",len(overlap))

Fold 0
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 1
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 2
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 3
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 4
Train subjects: 70
Val subjects: 17
Overlap: 0


In [2]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, BatchNormalization,
    AveragePooling2D, Dropout, SeparableConv2D, Flatten, Dense
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Model

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5

CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Filtered_8_30Hz"

# ✅ Best Optuna hyperparameters
best_params = {
    'f1_filters': 32,
    'depth_multiplier': 2,
    'dropout_rate': 0.416,
    'learning_rate': 0.000175,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================
df = pd.read_csv("dataset_metadata.csv")
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================
class EEGGenerator(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # fix shape (2560,27) -> (27,2560)
            data = data.T
            data = data[:, :, np.newaxis]
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# EEGNet Model
# =========================
def build_eegnet(f1_filters, depth_multiplier, dropout_rate, learning_rate):
    input_layer = Input(shape=(CHANNELS, TIMEPOINTS, 1))

    # Block 1
    block1 = Conv2D(f1_filters, (1, 64), padding="same", use_bias=False)(input_layer)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D(
        (CHANNELS, 1),
        depth_multiplier=depth_multiplier,
        use_bias=False,
        depthwise_constraint=max_norm(1.)
    )(block1)
    block1 = BatchNormalization()(block1)
    block1 = tf.keras.layers.Activation("elu")(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropout_rate)(block1)

    # Block 2
    f2_filters = f1_filters * depth_multiplier
    block2 = SeparableConv2D(f2_filters, (1, 16), use_bias=False, padding="same")(block1)
    block2 = BatchNormalization()(block2)
    block2 = tf.keras.layers.Activation("elu")(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropout_rate)(block2)

    # Classification layer
    flatten = Flatten()(block2)
    dense = Dense(N_CLASSES, activation="softmax", kernel_constraint=max_norm(0.25))(flatten)

    model = Model(inputs=input_layer, outputs=dense)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator(val_df, best_params['batch_size'], shuffle=False)

    model = build_eegnet(
        best_params['f1_filters'],
        best_params['depth_multiplier'],
        best_params['dropout_rate'],
        best_params['learning_rate']
    )

    # filenames include dataset name
    checkpoint_path = f"models/eegnet_{DATASET_NAME}_fold{fold + 1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/eegnet_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
515/515 [==============================] - 185s 357ms/step - loss: 0.5927 - accuracy: 0.6764 - val_loss: 0.5595 - val_accuracy: 0.7245 - lr: 1.7500e-04
Epoch 2/50
515/515 [==============================] - 30s 58ms/step - loss: 0.5010 - accuracy: 0.7609 - val_loss: 0.5094 - val_accuracy: 0.7498 - lr: 1.7500e-04
Epoch 3/50
515/515 [==============================] - 32s 62ms/step - loss: 0.4717 - accuracy: 0.7798 - val_loss: 0.4897 - val_accuracy: 0.7607 - lr: 1.7500e-04
Epoch 4/50
515/515 [==============================] - 31s 60ms/step - loss: 0.4539 - accuracy: 0.7879 - val_loss: 0.4676 - val_accuracy: 0.7716 - lr: 1.7500e-04
Epoch 5/50
515/515 [==============================] - 33s 63ms/step - loss: 0.4362 - accuracy: 0.8013 - val_loss: 0.4500 - val_accuracy: 0.7799 - lr: 1.7500e-04
Epoch 6/50
515/515 [==============================] - 33s 64ms/step - loss: 0.4229 - accuracy: 0.8070 

# Filtered 8-13

In [1]:
import numpy as np
import os

folder = "Filtered_8_13Hz"

sample_file = os.path.join(folder, os.listdir(folder)[0])

data = np.load(sample_file)

print("Sample shape:", data.shape)

Sample shape: (2560, 27)


In [2]:
import os
import pandas as pd

data_folder = "Filtered_8_13Hz"

files = []
labels = []
subjects = []

for fname in os.listdir(data_folder):
    if not fname.endswith(".npy"):
        continue

    path = os.path.join(data_folder, fname)

    parts = fname.split("_")
    subject = parts[2]

    if "Left" in fname:
        label = 0
    else:
        label = 1

    files.append(path)
    labels.append(label)
    subjects.append(subject)

df = pd.DataFrame({
    "file": files,
    "label": labels,
    "subject": subjects
})

print("Total samples:", len(df))
print("Total subjects:", df.subject.nunique())

df.to_csv("dataset_metadata_8_13Hz.csv", index=False)

Total samples: 20792
Total subjects: 87


In [3]:
from sklearn.model_selection import GroupKFold
import pandas as pd

df = pd.read_csv("dataset_metadata_8_13Hz.csv")

X = df["file"].values
y = df["label"].values
groups = df["subject"].values

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    train_subjects = set(groups[train_idx])
    val_subjects = set(groups[val_idx])

    overlap = train_subjects.intersection(val_subjects)

    print("Fold", fold)
    print("Train subjects:", len(train_subjects))
    print("Val subjects:", len(val_subjects))
    print("Overlap:", len(overlap))

Fold 0
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 1
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 2
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 3
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 4
Train subjects: 70
Val subjects: 17
Overlap: 0


In [1]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, BatchNormalization,
    AveragePooling2D, Dropout, SeparableConv2D, Flatten, Dense
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.models import Model

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5

CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Filtered_8_13Hz"

# ✅ Best Optuna hyperparameters
best_params = {
    'f1_filters': 32,
    'depth_multiplier': 2,
    'dropout_rate': 0.461,
    'learning_rate': 0.000231,
    'batch_size': 32
}

# =========================
# Load metadata
# =========================
df = pd.read_csv("dataset_metadata_8_13Hz.csv")
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator
# =========================
class EEGGenerator(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # fix shape (2560,27) -> (27,2560)
            data = data.T
            data = data[:, :, np.newaxis]
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# EEGNet Model
# =========================
def build_eegnet(f1_filters, depth_multiplier, dropout_rate, learning_rate):
    input_layer = Input(shape=(CHANNELS, TIMEPOINTS, 1))

    # Block 1
    block1 = Conv2D(f1_filters, (1, 64), padding="same", use_bias=False)(input_layer)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D(
        (CHANNELS, 1),
        depth_multiplier=depth_multiplier,
        use_bias=False,
        depthwise_constraint=max_norm(1.)
    )(block1)
    block1 = BatchNormalization()(block1)
    block1 = tf.keras.layers.Activation("elu")(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropout_rate)(block1)

    # Block 2
    f2_filters = f1_filters * depth_multiplier
    block2 = SeparableConv2D(f2_filters, (1, 16), use_bias=False, padding="same")(block1)
    block2 = BatchNormalization()(block2)
    block2 = tf.keras.layers.Activation("elu")(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropout_rate)(block2)

    # Classification layer
    flatten = Flatten()(block2)
    dense = Dense(N_CLASSES, activation="softmax", kernel_constraint=max_norm(0.25))(flatten)

    model = Model(inputs=input_layer, outputs=dense)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator(val_df, best_params['batch_size'], shuffle=False)

    model = build_eegnet(
        best_params['f1_filters'],
        best_params['depth_multiplier'],
        best_params['dropout_rate'],
        best_params['learning_rate']
    )

    # filenames include dataset name
    checkpoint_path = f"models/eegnet_{DATASET_NAME}_fold{fold + 1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)
    
    del model
    tf.keras.backend.clear_session()
    gc.collect()

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/eegnet_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
515/515 [==============================] - 48s 75ms/step - loss: 0.5830 - accuracy: 0.6845 - val_loss: 0.5564 - val_accuracy: 0.7210 - lr: 2.3100e-04
Epoch 2/50
515/515 [==============================] - 39s 75ms/step - loss: 0.4990 - accuracy: 0.7604 - val_loss: 0.5136 - val_accuracy: 0.7512 - lr: 2.3100e-04
Epoch 3/50
515/515 [==============================] - 38s 73ms/step - loss: 0.4709 - accuracy: 0.7803 - val_loss: 0.4792 - val_accuracy: 0.7686 - lr: 2.3100e-04
Epoch 4/50
515/515 [==============================] - 34s 67ms/step - loss: 0.4521 - accuracy: 0.7922 - val_loss: 0.4571 - val_accuracy: 0.7753 - lr: 2.3100e-04
Epoch 5/50
515/515 [==============================] - 36s 69ms/step - loss: 0.4360 - accuracy: 0.8009 - val_loss: 0.4325 - val_accuracy: 0.7941 - lr: 2.3100e-04
Epoch 6/50
515/515 [==============================] - 35s 68ms/step - loss: 0.4233 - accuracy: 0.8060 - 